# DM-AHR-ViT — results walkthrough

**Research prototype only. Not clinically validated, not for diagnosis or patient care.**

This notebook reads the artifacts already produced by the pipeline and explains what was
done and why. It needs **no GPU, no Modal account and no image download** — only the
repository and a Python environment with `pandas` and `matplotlib`.

Use it to walk a supervisor through the study: each section states the question, shows the
number that answers it, and names the file the number came from.

## 1. What was compared

The study asks whether generative (diffusion) hair removal helps a skin-lesion classifier
more than classical hair removal, on the public HAM10000 dermoscopy dataset.

| Arm | Input to the classifier | What it isolates |
|---|---|---|
| **A** | original image | baseline |
| **B** | DullRazor-style classical hair removal | classical hair removal |
| **C** | DM-AHR diffusion hair removal | generative hair removal |
| **A\*** | original resized 128→224 px | the resolution change Arm C introduces |

Arm A\* exists because DM-AHR works at 128 px. Without it, a drop in Arm C could be blamed
either on diffusion artefacts or on the resolution round-trip.

Three hypotheses were stated in advance:

- **H1** diffusion hair removal preserves lesion texture better than classical inpainting (PSNR/SSIM)
- **H2** the classifier does better on diffusion-cleaned images (macro-F1)
- **H3** explanations focus more on the lesion after diffusion removal (Grad-CAM, attention rollout)

These were hypotheses, not expectations. The results are reported as measured.

## 2. Environment for this notebook

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO = Path.cwd()
while not (REPO / "configs" / "protocol.json").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "configs" / "protocol.json").exists(), (
    "Run this notebook from inside the cloned DM-AHR-ViT repository."
)

ART = REPO / "artifacts"
LOGS = ART / "logs"
REP = REPO / "reports"
MAN = REPO / "data" / "manifests"


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}


def show(df, title=""):
    if title:
        print(title)
    return df


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
print("repository:", REPO)
print("manifests found:", MAN.exists(), "(split-level cells need them)")

## 3. The data

HAM10000 as distributed through the client's ISIC collection 212 contains **11,720** images —
more than the 10,015 the paper reports. The extra images carry no labels usable for this
study, so they are quarantined and never enter a split. Every count below is recomputed from
the manifest, not copied from the proposal.

In [ ]:
audit = jload(LOGS / "data_audit.json")

cohort = pd.DataFrame([
    ("Images in ISIC collection 212", audit.get("collection_total")),
    ("Confirmatory HAM10000 images (C0)", audit.get("c0_decoded")),
    ("Quarantined extras (C0-extra)", audit.get("c0_extra")),
    ("Lesion groups after dedup", audit.get("groups_total")),
    ("dHash candidate pairs examined", audit.get("dhash_candidates")),
    ("Near-duplicate pairs pixel-confirmed", audit.get("near_duplicates_confirmed")),
], columns=["quantity", "value"])
cohort

In [ ]:
fs = audit.get("frozen_splits", {})
counts = fs.get("counts", {})
if counts:
    tbl = pd.DataFrame(counts).T
    tbl.loc["TOTAL"] = tbl.sum()
    display(tbl)
else:
    print("Split counts not in data_audit.json; see reports/final_research_report.md")

print("\nSplit verification:", json.dumps(audit.get("split_verification", {}), indent=1)[:600])

In [ ]:
# Independent leakage check: no lesion group may appear in more than one split.
if (MAN / "splits.csv").exists():
    splits = pd.read_csv(MAN / "splits.csv")
    per_group = splits.groupby("group_id")["split"].nunique()
    leaking = per_group[per_group > 1]
    print("images:", len(splits))
    print("lesion groups:", splits["group_id"].nunique())
    print("groups spanning more than one split:", len(leaking))
    print("=> leakage-free:", len(leaking) == 0)
    print("\nImages per split:")
    print(splits["split"].value_counts().to_string())
else:
    print("data/manifests/splits.csv not present in this checkout.")
    print("The split is git-ignored by design (raw data stays out of Git).")
    print("Fetch it from the Modal volume, or read the verified result in")
    print("reports/final_research_report.md, section 'Cohort and splits'.")

## 4. Hair screening and the arms

A black-hat/top-hat morphological detector decides which images contain hair. That threshold
was **frozen before any test metric was computed** and tuned only on train/validation sources,
so it cannot have been fitted to the test set.

Only hair-present images differ between arms. A hair-absent image is byte-identical in A, B
and C — the removal step has nothing to remove.

In [ ]:
if (MAN / "hair_screening.csv").exists():
    hs = pd.read_csv(MAN / "hair_screening.csv")
    n = len(hs)
    present = int(hs["hair_present"].sum())
    print(f"images screened: {n}")
    print(f"hair-present:    {present} ({present / n:.1%})")
    print(f"hair-absent:     {n - present} ({(n - present) / n:.1%})")
    print("\nPer split:")
    print(hs.groupby("split")["hair_present"].agg(["size", "sum"]).to_string())
    print("\nscreening config hash:", hs["screen_config_hash"].dropna().iloc[0])
else:
    print("hair_screening.csv not present. The verified figure is 8,993 of 10,015")
    print("images (89.8%) screening as hair-present; see the report.")

In [ ]:
arms = pd.DataFrame([
    ("A", "original image", "none"),
    ("B", "DullRazor-style morphological detection + Telea inpainting", "hair-present images"),
    ("C", "DM-AHR conditional diffusion inpainting", "hair-present images"),
    ("A*", "original, bicubic 128 px round-trip", "all images (resolution control)"),
], columns=["arm", "preprocessing", "applied to"])
arms

## 5. Classification results

ImageNet-pretrained ViT-B/16, 7 classes, identical recipe for every arm, three seeds
(42, 123, 2026). The test split is touched once per frozen checkpoint.

Macro-F1 is the primary endpoint because the classes are severely imbalanced — accuracy is
dominated by `nv`, which is roughly two thirds of the data.

In [ ]:
seed = pd.read_csv(LOGS / "classification_seed_summary.csv")
full = seed[seed["subset"] == "full"].copy()
full = full.set_index("arm").loc[["A", "Astar", "B", "C"]].reset_index()
full[["arm", "macro_f1_mean", "macro_f1_sd", "accuracy_mean", "balanced_acc_mean",
      "auc_macro_mean", "n"]].round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(full["arm"], full["macro_f1_mean"], yerr=full["macro_f1_sd"],
          capsize=4, color=["#3b6ea5", "#8fa8bf", "#c47f3e", "#a4443f"])
ax[0].set_title("Test macro-F1 (mean ± SD over 3 seeds)")
ax[0].set_ylabel("macro-F1")
ax[0].set_ylim(0, 1)

ax[1].bar(full["arm"], full["auc_macro_mean"], color=["#3b6ea5", "#8fa8bf", "#c47f3e", "#a4443f"])
ax[1].set_title("Test macro AUC (mean over 3 seeds)")
ax[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()
print("Arm A* isolates the 128 px round-trip, so A vs A* is the resolution effect.")

In [ ]:
per_class = pd.read_csv(LOGS / "classification_per_class.csv")
for arm in ("A", "C"):
    sub = per_class[per_class["arm"] == arm]
    print(f"--- Arm {arm} per-class test metrics")
    display(sub[["class", "support", "f1_mean", "recall_mean",
                 "specificity_mean", "auc_mean"]].round(3).to_string(index=False))
print("'recall_mean' is the proposal's sensitivity. Supports below ~50 (df, vasc) are noisy.")

## 6. Image quality of the hair removal itself (H1)

PSNR and SSIM need a ground-truth hairless reference. DERMAHAIR pairs could not be obtained
(no Kaggle credentials), so the pipeline composites synthetic hair onto genuinely hair-absent
images, which gives a real reference. The hairy input is scored too, as a floor: a method that
scores below the unprocessed input has made the image worse.

In [ ]:
pq = pd.read_csv(LOGS / "paired_psnr_ssim_test.csv")
print("held-out pairs:", len(pq))
summary = pd.DataFrame({
    "PSNR": [pq["hairy_input_psnr"].mean(), pq["B_psnr"].mean(), pq["C_psnr"].mean()],
    "SSIM": [pq["hairy_input_ssim"].mean(), pq["B_ssim"].mean(), pq["C_ssim"].mean()],
}, index=["hairy input (floor)", "Arm B classical", "Arm C diffusion"])
summary.round(3)

## 7. Explainability (H3)

For every test image, Grad-CAM and attention roll-out are computed at the predicted class and
compared with the released HAM10000 lesion mask. Two numbers are reported: the fraction of the
top-20% salient pixels that land inside the mask, and a pointing game (does the single most
salient pixel hit the lesion). Chance baselines are computed alongside, because a mask that
covers most of the image would inflate both measures.

In [ ]:
import glob

xai = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(LOGS / "xai_localization_*.csv")))])
agg = (xai.groupby("arm")[["gradcam_overlap20", "rollout_overlap20",
                           "gradcam_pointing", "rollout_pointing",
                           "chance_uniform_overlap20", "chance_gaussian_overlap20"]]
       .mean().loc[["A", "Astar", "B", "C"]])
print("n images per arm:", xai.groupby("arm").size().to_dict())
agg.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
w = 0.35
idx = range(len(agg))
ax.bar([i - w / 2 for i in idx], agg["gradcam_overlap20"], w, label="Grad-CAM", color="#3b6ea5")
ax.bar([i + w / 2 for i in idx], agg["rollout_overlap20"], w, label="attention roll-out", color="#c47f3e")
ax.axhline(float(agg["chance_uniform_overlap20"].iloc[0]), ls="--", c="k", lw=1, label="uniform chance")
ax.set_xticks(list(idx)); ax.set_xticklabels(agg.index)
ax.set_ylabel("lesion overlap (top 20%)"); ax.set_title("Explanation alignment with lesion masks")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Statistical comparisons

Every endpoint is tested twice, as the protocol requires:

- **Bootstrap** — resamples whole lesion groups (not images) 10,000 times, the same resample
  shared across arms and seeds, giving a 95% CI and a p-value.
- **Wilcoxon signed-rank** — values collapsed to one number per lesion group first, so the many
  images of one lesion are not counted as independent evidence.

Holm correction is applied across the three planned pairwise comparisons within each endpoint,
separately for each test. A CI spanning zero is called inconclusive, not "no difference".

In [ ]:
stats = pd.read_csv(LOGS / "statistical_comparisons.csv")
print("comparisons:", len(stats), "| endpoints:", sorted(stats["endpoint"].unique()))
view = stats[["endpoint", "subset", "comparison", "difference", "ci_low", "ci_high",
              "p_holm", "effect_size_dz", "wilcoxon_p_raw", "wilcoxon_p_holm"]]
view.round(4)

In [ ]:
# Compact significance readout: both tests must agree before a result is called clear.
def verdict(r):
    if r["ci_low"] <= 0 <= r["ci_high"]:
        return "inconclusive"
    return "clear difference"

lines = []
for _, r in stats.iterrows():
    lines.append({
        "endpoint": r["endpoint"], "comparison": r["comparison"],
        "bootstrap": verdict(r),
        "wilcoxon_p_holm": round(float(r["wilcoxon_p_holm"]), 6),
        "difference": round(float(r["difference"]), 4),
    })
pd.DataFrame(lines)

## 9. What the results say about H1, H2 and H3

Answered from the table above, in plain words.

In [ ]:
def row(endpoint, comparison, subset=None):
    q = stats[(stats["endpoint"] == endpoint) & (stats["comparison"] == comparison)]
    if subset:
        q = q[q["subset"] == subset]
    return q.iloc[0] if len(q) else None

def phrase(r):
    if r is None:
        return "not computed"
    d, lo, hi = float(r["difference"]), float(r["ci_low"]), float(r["ci_high"])
    if lo <= 0 <= hi:
        return f"inconclusive (95% CI {lo:+.3f} to {hi:+.3f})"
    winner, loser = (r["arm_a"], r["arm_b"]) if d > 0 else (r["arm_b"], r["arm_a"])
    return f"{winner} higher than {loser} by {abs(d):.3f} (95% CI {lo:+.3f} to {hi:+.3f})"

print("H1 image quality, C vs B")
print("  PSNR:", phrase(row("paired_psnr", "C-B")))
print("  SSIM:", phrase(row("paired_ssim", "C-B")))
print("\nH2 classification, full test set")
print("  A vs C:", phrase(row("test_macro_f1", "A-C", "full")))
print("  B vs C:", phrase(row("test_macro_f1", "B-C", "full")))
print("\nH3 explainability")
print("  Grad-CAM overlap A vs C:  ", phrase(row("x1_gradcam_overlap20", "A-C")))
print("  attention overlap A vs C: ", phrase(row("x1_rollout_overlap20", "A-C")))
print("\nResolution control")
print("  A vs A*:", phrase(row("test_macro_f1", "A-Astar", "full")))

## 10. Visual panels (agent-reviewed, not expert-reviewed)

The pipeline writes contact sheets for the hair screening, the classical removal, the diffusion
smoke test and the explanation overlays. These were inspected by the agent during the run; they
have not been reviewed by a dermatologist.

In [ ]:
from IPython.display import Image, display

for name in ("hair_screening_panel.jpg", "dullrazor_panel.jpg", "dmahr_smoke_panel.jpg"):
    p = ART / "panels" / name
    if p.exists():
        print(name)
        display(Image(filename=str(p), width=760))
    else:
        print(name, "-> not in this checkout (panels are git-ignored; fetch from the volume)")

## 11. Acceptance gates

The project's own gate checker records what passed, and what is blocked with evidence.

In [ ]:
acc = jload(LOGS / "acceptance_report.json")
print("summary:", acc.get("summary"))
for g in acc.get("gates", []):
    print(f'{g["gate"]:>4}  {g["status"]:<7} {g["detail"][:90]}')
print()
for f in acc.get("functional_matrix", []):
    if f.get("status") != "PASS":
        print(f'{f["id"]}: {f["status"]} — {f["evidence"]}')

## 12. Limitations to state plainly, and what to say if asked

**The headline caveat.** Arm C is not DM-AHR as published. It was trained for 20,000 iterations
against 660,000 in the upstream config, sampled with 1,000 steps instead of 2,000 for budget
reasons, and runs at 128 px. Its weak result is a finding about an under-resourced model, not a
refutation of the method. Say this before showing the numbers, not after.

**Why the masks are not expert ground truth.** They are the released HAM10000 segmentations. The
source does not claim dermatologist tracing, so explanation overlap measures alignment with those
maps, not clinically correct reasoning.

**Why DERMAHAIR pairs are missing.** They are Kaggle-gated. Synthetic hair was composited onto
hair-absent images instead, which yields only 186 held-out pairs, so the paired intervals are wide.

**Why hair screening is unverified against real labels.** No independent hair annotations exist in
the data. The detector was calibrated on synthetic pairs and frozen; its 89.8% hair-present rate is
high and reported as a limitation.

**Licences.** HAM10000 is CC BY-NC 4.0. The DM-AHR repository contradicts itself — README says
CC BY-NC 4.0, LICENSE says Apache-2.0. Commercial use is unresolved and needs a rights decision.

### Likely supervisor questions, with short answers

| Question | Answer |
|---|---|
| Why not just report accuracy? | Classes are imbalanced; `nv` dominates. Macro-F1 is the primary endpoint, with balanced accuracy, per-class recall/specificity and AUC alongside. |
| How do you know the split is leakage-free? | Splits are grouped by lesion and confirmed near-duplicate, and the zero-overlap property is recomputed from `splits.csv` (section 3) and enforced by an automated test. |
| Why an extra A\* arm? | DM-AHR's 128 px resolution is a confound. A\* round-trips originals through the same resize so resolution and diffusion artefacts can be separated. |
| Aren't three seeds too few? | Yes — that is why inference is over 1,500 test images grouped into ~1,437 lesions, not over three seed means. Seeds are averaged within each bootstrap replicate, never counted as independent samples. |
| Why two statistical tests? | The proposal specifies Wilcoxon beside bootstrap CIs. They answer slightly different questions and are both reported with Holm correction. |
| Was the test set used for tuning? | No. The hair threshold and diffusion checkpoint were fit on train/validation only, and the test set was scored once per frozen checkpoint. |
| Why is Arm C so much worse? | Undertraining plus half the sampling budget plus 128 px upsampling. The report labels it an ablation and does not present it as the method's true performance. |
| What would a faithful rerun cost? | Arm C sampling alone is roughly 5.4 A100-hours at 1,000 steps and 10.9 at 2,000, before training and ViT runs. See `reports/compute_plan.md`. |